In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("nyu-mll/glue", "mrpc")
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)


def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [3]:
from transformers import TrainingArguments

training_args = TrainingArguments("test-trainer")
print("appareil :", training_args.device)
print("epochs :", training_args.num_train_epochs)
print("batch :", training_args.per_device_train_batch_size)
print("learning rate :", training_args.learning_rate)
print("log tous les :", training_args.logging_steps, "pas")

appareil : mps
epochs : 3.0
batch : 8
learning rate : 5e-05
log tous les : 500 pas


In [4]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 27443.50it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpo

In [5]:
from transformers import Trainer

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
)
print("batches par epoch :", len(trainer.get_train_dataloader()))

batches par epoch : 459


In [7]:
trainer.train()

/Users/magillie/dev/huggingface-nlp-course/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
500,0.514003
1000,0.394237


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.86it/s]
/Users/magillie/dev/huggingface-nlp-course/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.85it/s]


TrainOutput(global_step=1377, training_loss=0.42252562058171067, metrics={'train_runtime': 365.1435, 'train_samples_per_second': 30.136, 'train_steps_per_second': 3.771, 'total_flos': 405114969714960.0, 'train_loss': 0.42252562058171067, 'epoch': 3.0})

In [8]:
import numpy as np
import evaluate

def compute_metrics(eval_preds):
    metric = evaluate.load("glue", "mrpc")
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [9]:
faux_logits = np.array([[0.1, 0.9], [0.8, 0.2], [0.3, 0.7]])
print(compute_metrics((faux_logits, np.array([1, 0, 1]))))

{'accuracy': 1.0, 'f1': 1.0}


In [11]:
predictions = trainer.predict(tokenized_datasets["validation"])
print(predictions.predictions.shape, predictions.label_ids.shape)

preds = np.argmax(predictions.predictions, axis=-1)
print("predictions par classe :", np.bincount(preds))
print("vrais labels par classe :", np.bincount(predictions.label_ids))

(408, 2) (408,)
predictions par classe : [102 306]
vrais labels par classe : [129 279]


In [12]:
metric = evaluate.load("glue", "mrpc")
print(metric.compute(predictions=preds, references=predictions.label_ids))
print(predictions.metrics)

{'accuracy': 0.8259803921568627, 'f1': 0.8786324786324786}
{'test_loss': 0.6193695664405823, 'test_runtime': 1.3422, 'test_samples_per_second': 303.974, 'test_steps_per_second': 37.997}


In [13]:
training_args = TrainingArguments("test-trainer", eval_strategy="epoch")
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 23502.46it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpo

In [14]:
trainer.train()

/Users/magillie/dev/huggingface-nlp-course/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.408217,0.803922,0.870968
2,0.533025,0.510875,0.845588,0.891192
3,0.313758,0.633822,0.857843,0.902685


/Users/magillie/dev/huggingface-nlp-course/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.92it/s]
/Users/magillie/dev/huggingface-nlp-course/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/magillie/dev/huggingface-nlp-course/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.99it/s]
/Users/magillie/dev/huggingface-nlp-course/.venv/lib/python3.11/site-packages/torch/utils/dat

TrainOutput(global_step=1377, training_loss=0.3575573757063589, metrics={'train_runtime': 360.912, 'train_samples_per_second': 30.489, 'train_steps_per_second': 3.815, 'total_flos': 405114969714960.0, 'train_loss': 0.3575573757063589, 'epoch': 3.0})